In [1]:
# Importing libraries
import pandas as pd
import numpy as np

# ── STEP 1: Load all data ───────────────────────────────────────

# Credit snapshots
snap_files = {
    'jan': './datasets/Credit Data - 01-01-2025.csv',
    'mar': './datasets/Credit Data - 30-03-2025.csv',
    'jun': './datasets/Credit Data - 30-06-2025.csv',
    'sep': './datasets/Credit Data - 30-09-2025.csv',
    'dec': './datasets/Credit Data - 30-12-2025.csv'
}
snap_dates = {
    'jan': pd.Timestamp('2025-01-01'),
    'mar': pd.Timestamp('2025-03-30'),
    'jun': pd.Timestamp('2025-06-30'),
    'sep': pd.Timestamp('2025-09-30'),
    'dec': pd.Timestamp('2025-12-30'),
}

# Load and stack all snapshots into one table
snaps = []
for name, path in snap_files.items():
    df = pd.read_csv(path)
    df['snapshot'] = name
    df['snapshot_date'] = snap_dates[name]
    snaps.append(df)
credit = pd.concat(snaps, ignore_index=True)

# ── STEP 2: Load demographics ───────────────────────────────────

xl_path = './datasets/Sales and Customer Data.xlsx'

dob_df = pd.read_excel(xl_path, sheet_name='DOB').dropna(how='all')
income_df = pd.read_excel(xl_path, sheet_name='Income Level').dropna(how='all')
gender_df = pd.read_excel(xl_path, sheet_name='Gender').dropna(how='all')
sales_df = pd.read_excel(xl_path, sheet_name='Sales Details').dropna(how='all')

# Fix DOB column name (has trailing space)
dob_df = dob_df.rename(columns={'Loan Id ': 'Loan Id'})

# ── STEP 3: Clean DOB and calculate age ────────────────────────

# Parse date — strip timezone info
dob_df['dob'] = pd.to_datetime(dob_df['date_of_birth'], errors='coerce', utc=True)
dob_df['dob'] = dob_df['dob'].dt.tz_localize(None)

# Drop rows with no Loan Id (can't link them)
dob_df = dob_df.dropna(subset=['Loan Id'])

# Keep only most recent DOB record per loan (in case of duplicates)
dob_df = dob_df.sort_values('createdAt UTC', ascending=False)
dob_clean = dob_df.drop_duplicates(subset='Loan Id', keep='first')[['Loan Id','dob']]

print(f"DOB records with valid Loan Id: {len(dob_clean):,}")
print(f"DOB records with valid date: {dob_clean['dob'].notnull().sum():,}")

# ── STEP 4: Age calculation function ───────────────────────────

def calculate_age(dob, snapshot_date):
    if pd.isnull(dob):
        return np.nan
    age = snapshot_date.year - dob.year
    if (snapshot_date.month, snapshot_date.day) < (dob.month, dob.day):
        age -= 1
    return age

# ── STEP 5: Apply age per snapshot ─────────────────────────────

for snap_name, snap_date in snap_dates.items():
    dob_clean[f'age_{snap_name}'] = dob_clean['dob'].apply(
        lambda d: calculate_age(d, snap_date)
    )

# ── STEP 6: Flag impossible ages ───────────────────────────────
# Under 18 or over 80 = data quality issue, exclude from analysis

for snap_name in snap_dates.keys():
    col = f'age_{snap_name}'
    invalid = (dob_clean[col] < 18) | (dob_clean[col] > 80)
    dob_clean.loc[invalid, col] = np.nan  # set to null, don't delete

print(f"\nInvalid ages excluded (under 18 or over 80): {invalid.sum()}")

# ── STEP 7: Create age bands ────────────────────────────────────

age_bins   = [17, 25, 35, 45, 55, 999]
age_labels = ['18-25', '26-35', '36-45', '46-55', '55+']

for snap_name in snap_dates.keys():
    dob_clean[f'age_band_{snap_name}'] = pd.cut(
        dob_clean[f'age_{snap_name}'],
        bins=age_bins,
        labels=age_labels,
        right=True
    )

print("\nAge band distribution (as at Jan 2025):")
print(dob_clean['age_band_jan'].value_counts().sort_index())

# ── STEP 8: Average monthly income ─────────────────────────────

income_df = income_df.dropna(subset=['Loan Id'])
income_df['avg_monthly_income'] = income_df['Received'] / income_df['Duration']

# Remove impossible values
income_df.loc[income_df['avg_monthly_income'] < 0, 'avg_monthly_income'] = np.nan
income_df.loc[income_df['avg_monthly_income'] > 5_000_000, 'avg_monthly_income'] = np.nan

income_bins   = [0, 4999, 9999, 19999, 29999, 49999, 99999, 149999, float('inf')]
income_labels = ['Below 5K','5K-9.9K','10K-19.9K','20K-29.9K',
                 '30K-49.9K','50K-99.9K','100K-149.9K','150K+']

income_df['income_band'] = pd.cut(
    income_df['avg_monthly_income'],
    bins=income_bins, labels=income_labels, right=True
)

print("\nIncome band distribution:")
print(income_df['income_band'].value_counts().sort_index())
# ── STEP 9: Build master demographics table ─────────────────────

demographics = dob_clean.merge(
    income_df[['Loan Id','avg_monthly_income','income_band']],
    on='Loan Id', how='left'
).merge(
    gender_df[['Loan Id','Gender']],
    on='Loan Id', how='left'
)

print(f"\nDemographics master table: {len(demographics):,} rows")
print(f"With income data: {demographics['avg_monthly_income'].notnull().sum():,}")
print(f"With gender data: {demographics['Gender'].notnull().sum():,}")

# ── STEP 10: Merge into credit snapshots ────────────────────────

master = credit.merge(
    demographics.rename(columns={'Loan Id': 'LOAN_ID'}),
    on='LOAN_ID', how='left'
)

# Pick the right age band per row based on snapshot
master['age_band'] = master.apply(
    lambda row: row.get(f"age_band_{row['snapshot']}", np.nan), axis=1
)

# Check match rates
total = len(credit)
with_age = master['dob'].notnull().sum()
with_income = master['avg_monthly_income'].notnull().sum()
print(f"\n── MASTER TABLE MATCH RATES ──")
print(f"Total credit records: {total:,}")
print(f"Records with age data: {with_age:,} ({with_age/total:.1%})")
print(f"Records with income data: {with_income:,} ({with_income/total:.1%})")

DOB records with valid Loan Id: 11,217
DOB records with valid date: 11,175

Invalid ages excluded (under 18 or over 80): 4

Age band distribution (as at Jan 2025):
age_band_jan
18-25    3159
26-35    5637
36-45    1777
46-55     502
55+        97
Name: count, dtype: int64

Income band distribution:
income_band
Below 5K        301
5K-9.9K         696
10K-19.9K      1634
20K-29.9K      1461
30K-49.9K      1953
50K-99.9K      2702
100K-149.9K    1238
150K+          1881
Name: count, dtype: int64

Demographics master table: 17,867 rows
With income data: 16,597
With gender data: 16,404

── MASTER TABLE MATCH RATES ──
Total credit records: 71,456
Records with age data: 42,677 (59.7%)
Records with income data: 38,462 (53.8%)
